# 低延迟视觉增强终端 — 交互控制面板

这个面板直接驱动 `enhance_dma.bit`：**每一帧都真的经过 FPGA 内的 `enhance` 核**，
图里显示的是 AXI-DMA 读回来的硬件输出，不是 numpy 模拟。

点 **Run** 看当前预设，点 **Benchmark x20** 取 20 帧里最好的一次。
右下角三行是这轮的全部证据：

- **frame**：整帧往返时间（含 DMA 与 PS 开销）
- **latency**：内核流水线填充延迟（内核前后各打一个时间戳相减）
- **bit-exact**：与 numpy 黄金模型逐通道比对的结果，正常是 `0 / 2764800`


In [ ]:
from enhance_panel import EnhancePanel

panel = EnhancePanel()          # 首次点击时才加载 overlay
panel.show()


## 一次跑完 8 个预设（录视频时用这一格）

输出是一张表：每行一个预设的耗时、延迟与逐位比对结果。
把上面面板里的预设逐个点一遍、下面这张表一并录进视频，就同时演示了
**参数化**（寄存器实时改）与**正确性**（零容差比对）。


In [ ]:
rows = []
for name in panel.presets:
    r = panel.apply(name, frames=2)
    rows.append((name, r['frame_ms'], r['latency_cycles'], r['mismatched']))

print(f"{'preset':<10}{'frame_ms':>10}{'latency':>10}{'mismatched':>12}")
for name, ms, lat, bad in rows:
    print(f'{name:<10}{ms:>10.2f}{lat:>10}{bad:>12}')


## 说明与边界

- 这里走的是 **DMA 通路**，所以不需要显示器和采集卡；
- **latency** 是内核流水线延迟（实测 16 个像素时钟 = 160 ns @100 MHz），
  不是端到端延迟——端到端还要加上 HDMI 输入/输出链路，需要采集卡才能测；
- `bit-exact` 用 `wrap` 边界（与 Sobel 参考图同一约定），因为硬件行缓冲里
  保存的是上一帧的行；
- 换个预设时，内核最多可能有一帧看到新旧参数混合，下一帧即正确。
